# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [ ]:
! pip3 install -q schedule pytest requests beautifulsoup4 # установка библиотек, если ещё не

In [4]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [7]:
 def get_book_data(book_url: str) -> dict:
    """
    Собирает информацию об одной книге со страницы каталога books.toscrape.com.

    Args:
        book_url (str): Полный URL страницы книги.

    Returns:
        dict: Словарь с полями title, price, rating, availability,
              description и product_information.
    """
    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    response = requests.get(book_url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')

    title_tag = soup.find('h1')
    title = title_tag.get_text(strip=True) if title_tag else None

    price_tag = soup.find('p', class_='price_color')
    price = price_tag.get_text(strip=True) if price_tag else None

    rating_tag = soup.find('p', class_='star-rating')
    rating = None
    if rating_tag:
        for cls in rating_tag.get('class', []):
            if cls != 'star-rating':
                rating = cls
                break

    availability_tag = soup.find('p', class_='instock availability')
    availability = availability_tag.get_text(strip=True) if availability_tag else None

    desc_header = soup.find('div', id='product_description')
    description = None
    if desc_header:
        desc_p = desc_header.find_next_sibling('p')
        if desc_p:
            description = desc_p.get_text(strip=True)

    product_info = {}
    table = soup.find('table', class_='table-striped')
    if table:
        for row in table.find_all('tr'):
            th = row.find('th')
            td = row.find('td')
            if th and td:
                product_info[th.get_text(strip=True)] = td.get_text(strip=True)

    return {
        'title': title,
        'price': price,
        'rating': rating,
        'availability': availability,
        'description': description,
        'product_information': product_info
    }
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [8]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': 'Â£51.77',
 'rating': 'Three',
 'availability': 'In stock (22 available)',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [9]:
def scrape_books(is_save: bool = False) -> list:
    """
    Обходит все страницы каталога books.toscrape.com и собирает данные о книгах.

    Args:
        is_save (bool): Если True, сохраняет результат в books_data.txt.

    Returns:
        list[dict]: Список словарей с данными о книгах.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    base_url = 'http://books.toscrape.com/catalogue/page-{}.html'
    all_books = []
    page = 1

    while True:
        url = base_url.format(page)
        response = requests.get(url, timeout=10)
        
        if response.status_code != 200:
            break
            
        soup = BeautifulSoup(response.text, 'html.parser')
        book_links = soup.select('article.product_pod h3 a')
        
        if not book_links:
            break
            
        for link in book_links:
            relative_url = link['href']
            book_url = f'http://books.toscrape.com/catalogue/{relative_url}'
            try:
                book_data = get_book_data(book_url)
                all_books.append(book_data)
            except Exception as e:
                print(f'Ошибка при парсинге {book_url}: {e}')
                
        page += 1
        time.sleep(0.5)

    if is_save:
        with open('books_data.txt', 'w', encoding='utf-8') as f:
            for book in all_books:
                f.write(str(book) + '\n')
        print(f'Сохранено {len(all_books)} книг в books_data.txt')

    return all_books
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [10]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

Сохранено 1000 книг в books_data.txt
<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
def scheduled_scrape():
    scrape_books(is_save=True)

schedule.every().day.at('19:00').do(scheduled_scrape)
print('Планировщик запущен. Ожидание 19:00...')
while True:
    schedule.run_pending()
    time.sleep(60)  
# КОНЕЦ ВАШЕГО РЕШЕНИЯ
# была проведена проверка в 19:55, так как цикл бесконечный, пришлось приостановить работу

Планировщик запущен. Ожидание 19:55...


KeyboardInterrupt: 

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [1]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
!pytest tests/test_scraper.py -v

============================= test session starts ==============================
platform darwin -- Python 3.14.5, pytest-9.2.0, pluggy-1.7.0 -- /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14
cachedir: .pytest_cache
rootdir: /Users/repinaalenamaksimovna/PythonProjects/Готово/HW3
plugins: platformdirs-4.12.3
collected 3 items                                                              

tests/test_scraper.py::test_get_book_data_returns_dict_with_expected_keys PASSED [ 33%]
tests/test_scraper.py::test_get_book_data_title_is_correct PASSED        [ 66%]
tests/test_scraper.py::test_scrape_books_returns_non_empty_list PASSED   [100%]

======================== 3 passed in 515.53s (0:08:35) =========================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Было задано 5 здач, две из которых мне показались нестандартными. Думаю, я выполнила все задания, возможно с ошибками, но в любом случае результат представлен так, как я его видела у себя в голове...`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Я выполняла задания долго, как и всегда, сначала была проблема в установке библиотек, выдавал ошибку, по совету ИИ заменила pip на pip3. Самыми простыми мне показались задания 1 и 2, однако во 2 задании я переживала, что код долго не может запуститься, до этого у меня никогда не было, чтобы на запуск уходило более 8 минут, поэтому сначала я подумала, что допущена какая-то ошибка, и всё просто виснет. Трудности начались с задания 3, где нужно было настроить автоматический запуск функции сбора данных каждый день в одно и то же время, я до сих пор не знаю, в правильном ли виде сделала задание, так как это бесконечный цикл, было ощущение, что у меня всё сломалось...Для теста я выбрала ближайшее ко мне время, чтобы был вывод на экран, а постом исправила согласно условию задачи и остановила запуск, поэтому возникает ошибка. Возможно это можно было сделать более правильно, но я не догадалась как, поэтому решила оставить в таком виде. Далее в задании 4 нужно было создать в одной папке файлы определенного формата, тоже не привыкла к этому, учитывая, что у меня был опыт выполнения только заданий в рамках нашего курса по программированию. Однако в принципе мне помогло объясенение в задании и подсказки от ИИ. Самым трудным заданием было для меня настроить GitHub, тут я плотно пользоваласт ИИ, так как мне трудно было и непривычно работать в терминале и задавать верные команды. Плюс у меня не было других проектов для сранения в последних пунктах, поэтому я просто взята то же самое задание и немного его откорректировала, чтобы получилось сравнить. Также у меня были лишние файлы в репозитории, которые я потом смогла убрать с помощью терминала, но с подсказками ИИ..`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: В основном я использовала ИИ, как обычно, для выяснения причин возникновения ошибки плюс для составления инструкции по работе с GitHub. Например, при установке библиотеки выдавало ошибку command not found: pip, поэтому я отправила скрин для получения совета по решению проблемы и выснения причин, в итоге исправила на pip3. также я уточняла у ИИ, норально ли, что код грузится уже 8 минут, получила ответ, что это нормально для такого объёма данных и. успокоилась. В большей степени я использовала ИИ для последнего задания, так как у меня часто возникали ошибки, и мне тяжело было работать с терминалом, ИИ давал мне инструкции, когда я получала не те результаты, что хотела, и справшивала, как выйти из ситуации. Без ИИ, думаю, я мне было бы очень сложно справиться с заданием. Однако, мне кажется, что я много чего узнала благодаря этой инструкции`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: На самом деле, думаю, что я проснила в ходе выполнения заданий для себя многие моменты, которые раньше не понимала, поэтому нет чётко сформулированных вопросов...`

**5. Самооценка по критериям задания**:

`Ответ: 7-8/10...`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Спасибо, что даёте порботать над интересными задачами...`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

